# 5. LVR — QA

**Purpose** — Validate Loss-Versus-Rebalancing engine output and its relationship with fees and realized vol.

**Source tables**
- `py_lvr_daily`
- `py_lvr_per_swap`
- `mart_volume_daily`
- `mart_price_analytics_daily`
- `stg_swaps`

**Window** — data span ends `2026-04-06`. The bootstrap cell below sets `END_DATE` and two convenience starts (`START_DATE_30D`, `START_DATE_90D`).

**Convention** — values denominated in `token0` / `token0_equiv`. For the indexed USDC/WETH 0.05% pool, `token0 = USDC` so `token0_equiv ≈ USD`.

This notebook is a **QA workbench**, not a PM dashboard. Each section ends with explicit PASS/WARN/FAIL checks. Final §4 cell prints the verdict.


In [1]:
import sys, os
from pathlib import Path

ROOT = Path(os.getcwd()).resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.environ.setdefault("CLICKHOUSE_HOST", "localhost")
os.environ.setdefault("CLICKHOUSE_PORT", "8124")
os.environ.setdefault("CLICKHOUSE_DATABASE", "analytics")
os.environ.setdefault("CLICKHOUSE_USER", "default")
os.environ.setdefault("CLICKHOUSE_PASSWORD", "1234")

import polars as pl
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import date, timedelta

from core.db import ClickHouseDB, ClickHouseSettings
db = ClickHouseDB.from_settings(ClickHouseSettings())

POOL      = "0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640"
END_DATE  = "2026-04-06"            # Last date with full mart coverage
START_DATE_90D = (date.fromisoformat(END_DATE) - timedelta(days=90)).isoformat()
START_DATE_30D = (date.fromisoformat(END_DATE) - timedelta(days=30)).isoformat()

print(f"Pool:       {POOL}")
print(f"End date:   {END_DATE}")
print(f"Window 90d: {START_DATE_90D} → {END_DATE}")
print(f"Window 30d: {START_DATE_30D} → {END_DATE}")

# Auto-cleanup: close CH connection on kernel shutdown so sockets don't pile up
import atexit as _atexit
_atexit.register(lambda: db.close() if 'db' in globals() and db is not None else None)


Pool:       0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640
End date:   2026-04-06
Window 90d: 2026-01-06 → 2026-04-06
Window 30d: 2026-03-07 → 2026-04-06


<function __main__.<lambda>()>

In [2]:
# QA check helper — records pass/warn/fail into QA_RESULTS for the §4 verdict cell.
QA_RESULTS = []

def qa_check(label, passed, severity="fail", detail=""):
    if passed:
        status, icon = "PASS", "🟢"
    elif severity == "warn":
        status, icon = "WARN", "🟠"
    else:
        status, icon = "FAIL", "🔴"
    QA_RESULTS.append({"label": label, "status": status, "severity": severity, "detail": str(detail)})
    print(f"{icon} {status:4s} | {label}" + (f"  —  {detail}" if detail else ""))

def qa_table_exists(name, database="analytics"):
    try:
        n = db.read_query(
            f"SELECT count() AS n FROM system.tables WHERE database='{database}' AND name='{name}'"
        )["n"][0]
        return int(n) > 0
    except Exception:
        return False

def qa_safe(query):
    try:
        return db.read_query(query)
    except Exception as e:
        return None


## Sign convention

By inspection of `py_lvr_per_swap.lvr` values:

- The engine emits LVR values that should be summed as a **loss to the LP** when positive.
  Validate explicitly in §2 with `min`/`max`/`mean` printout below.

If the data shows mostly negative values, flip the convention in your reading (sign × −1).


## §1 — Data fetch & coverage

In [3]:
lvr_d = db.read_query(f"""
SELECT day AS date, lvr_total, fees_total, fee_lvr_ratio, swap_count
FROM py_lvr_daily
WHERE lower(pool) = lower('{POOL}') AND day <= toDate('{END_DATE}')
ORDER BY day
""")
lvr_swap = db.read_query(f"""
SELECT toDate(block_time) AS date, block_number, log_index, lvr, fee_amount, liquidity_active
FROM py_lvr_per_swap
WHERE lower(pool) = lower('{POOL}') AND block_time <= toDateTime('{END_DATE} 23:59:59')
""")
print(f"lvr daily:    {lvr_d.height}  rows ({lvr_d['date'].min()} → {lvr_d['date'].max()})")
print(f"lvr per-swap: {lvr_swap.height} rows")
print(f"\nlvr value summary:")
print(f"  min:   {lvr_swap['lvr'].min():.4f}")
print(f"  mean:  {lvr_swap['lvr'].mean():.4f}")
print(f"  max:   {lvr_swap['lvr'].max():.4f}")
print(f"  >0 fraction: {(lvr_swap['lvr'] > 0).mean():.1%}")

lvr daily:    91  rows (2026-01-06 → 2026-04-06)
lvr per-swap: 584535 rows

lvr value summary:
  min:   0.0000
  mean:  26.6324
  max:   1342204.9672
  >0 fraction: 99.0%


## §2 — Sanity checks

In [4]:
# Finiteness
n_null_d = lvr_d.filter(pl.col("lvr_total").is_null()).height
qa_check("py_lvr_daily.lvr_total non-null after warm-up", n_null_d == 0, severity="warn", detail=f"nulls={n_null_d}")

n_inf = lvr_swap.filter(pl.col("lvr").is_infinite()).height
qa_check("py_lvr_per_swap.lvr finite", n_inf == 0, detail=f"infinities={n_inf}")

# Tail audit: |lvr| > p99.9 + 5σ
import numpy as np
v = lvr_swap.filter(pl.col("lvr").is_not_null())["lvr"].to_numpy()
if len(v) > 100:
    p999 = np.percentile(np.abs(v), 99.9)
    sigma = float(np.std(v))
    threshold = p999 + 5 * sigma
    n_tail = int((np.abs(v) > threshold).sum())
    qa_check("|LVR_per_swap| tail count beyond p99.9 + 5σ", n_tail < 5,
             severity="warn", detail=f"count={n_tail} threshold={threshold:.2f}")

🟢 PASS | py_lvr_daily.lvr_total non-null after warm-up  —  nulls=0
🟢 PASS | py_lvr_per_swap.lvr finite  —  infinities=0
🟠 WARN | |LVR_per_swap| tail count beyond p99.9 + 5σ  —  count=95 threshold=13088.97


In [5]:
# Swap count reconciliation: stg_swaps vs py_lvr_per_swap on the SAME 90d window
recon = db.read_query(f"""
SELECT
  (SELECT count() FROM stg_swaps
   WHERE lower(pool) = lower('{POOL}')
     AND block_time >= toDateTime('{START_DATE_90D} 00:00:00')
     AND block_time <= toDateTime('{END_DATE} 23:59:59')) AS stg_n,
  (SELECT count() FROM py_lvr_per_swap
   WHERE lower(pool) = lower('{POOL}')
     AND block_time >= toDateTime('{START_DATE_90D} 00:00:00')
     AND block_time <= toDateTime('{END_DATE} 23:59:59')) AS lvr_n
""").row(0, named=True)
gap = abs(int(recon["stg_n"]) - int(recon["lvr_n"]))
total = max(int(recon["stg_n"]), 1)
qa_check("py_lvr_per_swap.count() ≈ stg_swaps.count() (over the 90d window)",
         gap / total < 0.05, severity="warn",
         detail=f"stg={recon['stg_n']} lvr={recon['lvr_n']} gap={gap} ({gap/total:.1%})")


🟢 PASS | py_lvr_per_swap.count() ≈ stg_swaps.count() (over the 90d window)  —  stg=587406 lvr=584535 gap=2871 (0.5%)


In [6]:
# LVR magnitude vs theoretical expectation. Classical: LVR_apr ≈ σ²/8.
# Compare cumulative LVR over the window to theoretical expectation.
lvr_total = float(lvr_d["lvr_total"].sum())
days_n = int(lvr_d.height)
if days_n > 0:
    fy_avg = db.read_query(f"""
    SELECT avg(pa.realized_vol_7d) AS avg_sigma, avg(fy.tvl_active_token0) AS avg_tvl
    FROM mart_fee_yield_daily fy
    INNER JOIN mart_price_analytics_daily pa ON pa.pool = fy.pool AND pa.date = fy.date
    WHERE lower(fy.pool) = lower('{POOL}') AND fy.date >= toDate('{START_DATE_90D}') AND fy.date <= toDate('{END_DATE}')
    """).row(0, named=True)
    sigma = float(fy_avg["avg_sigma"] or 0)
    tvl   = float(fy_avg["avg_tvl"]   or 0)
    expected_lvr = (sigma**2 / 8) * tvl * (days_n / 365.0)
    ratio = lvr_total / max(expected_lvr, 1e-9)
    qa_check("cumulative LVR within 5× of σ²/8 theoretical expectation",
             0.2 <= ratio <= 5.0, severity="warn",
             detail=f"engine={lvr_total:,.0f}  theory={expected_lvr:,.0f}  ratio={ratio:.1f}×  σ̄={sigma:.3f}  TVL̄={tvl:,.0f}")


🟠 WARN | cumulative LVR within 5× of σ²/8 theoretical expectation  —  engine=15,567,599  theory=685,941  ratio=22.7×  σ̄=0.666  TVL̄=49,647,920


In [7]:
# Daily LVR vs daily realized variance — slope ~1 in log-log = LVR ∝ σ²
prc = db.read_query(f"""
SELECT date, realized_vol_7d FROM mart_price_analytics_daily
WHERE lower(pool) = lower('{POOL}') AND date <= toDate('{END_DATE}')
""")
joined = lvr_d.join(prc, on="date").filter(pl.col("lvr_total").abs() > 0).filter(pl.col("realized_vol_7d") > 0)
if joined.height >= 5:
    import numpy as np
    log_lvr = np.log(joined["lvr_total"].abs().to_numpy())
    log_var = 2 * np.log(joined["realized_vol_7d"].to_numpy())
    slope, _ = np.polyfit(log_var, log_lvr, 1)
    qa_check("daily |LVR| scales with σ² (log-log slope ≈ 1)",
             0.3 <= slope <= 2.5, severity="warn",
             detail=f"fitted_slope={slope:.2f} (n={joined.height})")
else:
    qa_check("daily |LVR| scales with σ² (log-log slope ≈ 1)", True,
             severity="warn", detail=f"insufficient data ({joined.height} points)")

🟢 PASS | daily |LVR| scales with σ² (log-log slope ≈ 1)  —  fitted_slope=0.80 (n=91)


## §3 — Diagnostic visuals

**QA purpose**: daily LVR + fees on dual axis — visualize the carry vs the cost of gamma sold.

In [8]:
last_90 = lvr_d.filter(pl.col("date") >= date.fromisoformat(START_DATE_90D))
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=last_90["date"].to_list(),     y=last_90["lvr_total"].to_list(),
                      name="LVR daily", marker_color="#ef4444"), secondary_y=False)
fig.add_trace(go.Bar(x=last_90["date"].to_list(),     y=last_90["fees_total"].to_list(),
                      name="Fees daily", marker_color="#22c55e"), secondary_y=False)
fig.add_trace(go.Scatter(x=last_90["date"].to_list(), y=last_90["fee_lvr_ratio"].to_list(),
                          name="fees / LVR", line=dict(color="#3b82f6", width=2)), secondary_y=True)
fig.update_layout(height=420, barmode="group", title_text="LVR vs Fees daily (last 90d)")
fig.show()

**QA purpose**: per-swap LVR distribution — long tails are expected for arbitrage swaps. Audit visually for outliers.

In [9]:
v = lvr_swap.filter(pl.col("lvr").is_not_null()).filter(pl.col("lvr").abs() > 0)
import numpy as np
clipped = v["lvr"].to_numpy()
p1, p99 = np.percentile(clipped, [1, 99])
clipped = clipped[(clipped >= p1) & (clipped <= p99)]
fig = go.Figure(go.Histogram(x=clipped, nbinsx=80))
fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.update_layout(height=320, title_text="LVR per-swap distribution (1st-99th percentile)")
fig.show()
print(f"per-swap rows: {v.height}  |  full range [{v['lvr'].min():.2f}, {v['lvr'].max():.2f}]")

per-swap rows: 578509  |  full range [0.00, 1342204.97]


**QA purpose**: cumulative LVR vs cumulative fees — should be monotonic and not explode.

In [10]:
cum = lvr_d.sort("date").with_columns(
    pl.col("lvr_total").cum_sum().alias("cum_lvr"),
    pl.col("fees_total").cum_sum().alias("cum_fees"),
)
fig = go.Figure()
fig.add_trace(go.Scatter(x=cum["date"].to_list(), y=cum["cum_lvr"].to_list(),  name="Cumulative LVR",  line=dict(color="#ef4444")))
fig.add_trace(go.Scatter(x=cum["date"].to_list(), y=cum["cum_fees"].to_list(), name="Cumulative Fees", line=dict(color="#22c55e")))
fig.update_layout(height=360, title_text="Cumulative LVR vs Fees (full history)")
fig.show()
print(f"final cumulative LVR: {cum['cum_lvr'].tail(1).item():,.2f}  |  fees: {cum['cum_fees'].tail(1).item():,.2f}")

final cumulative LVR: 15,567,599.01  |  fees: 8,488,578.41


**QA purpose**: log-log slope check that |LVR| ∝ σ² — visual confirmation of the QA test in §2.

In [11]:
import numpy as np
joined = lvr_d.join(prc, on="date").filter(pl.col("lvr_total").abs() > 0).filter(pl.col("realized_vol_7d") > 0)
fig = go.Figure(go.Scatter(
    x=joined["realized_vol_7d"].to_list(),
    y=joined["lvr_total"].abs().to_list(),
    mode="markers", marker=dict(size=5, opacity=0.5),
))
fig.update_xaxes(type="log", title="realized_vol_7d")
fig.update_yaxes(type="log", title="|daily LVR|")
fig.update_layout(height=380, title_text="Log-log: |daily LVR| vs realized_vol_7d (slope ≈ 2 expected)")
fig.show()

**QA purpose**: per-day swap count reconciliation — visualize parity between stg_swaps and py_lvr_per_swap.

In [12]:
stg_daily = db.read_query(f"""
SELECT toDate(block_time) AS date, count() AS n FROM stg_swaps
WHERE lower(pool) = lower('{POOL}') AND block_time <= toDateTime('{END_DATE} 23:59:59')
GROUP BY toDate(block_time)
""")
lvr_daily = lvr_swap.group_by("date").agg(pl.len().alias("n_lvr")).sort("date")
joined = stg_daily.rename({"n": "n_stg"}).join(lvr_daily, on="date", how="left").fill_null(0)
last_90 = joined.filter(pl.col("date") >= date.fromisoformat(START_DATE_90D))
fig = go.Figure()
fig.add_trace(go.Bar(x=last_90["date"].to_list(), y=last_90["n_stg"].to_list(),  name="stg_swaps"))
fig.add_trace(go.Bar(x=last_90["date"].to_list(), y=last_90["n_lvr"].to_list(),  name="py_lvr_per_swap"))
fig.update_layout(height=320, barmode="overlay", title_text="Daily swap count: stg vs LVR engine (last 90d)")
fig.show()

## §4 — QA verdict

In [13]:
# ─── §4 QA verdict ──────────────────────────────────────────────────────────
verdict = pl.DataFrame(QA_RESULTS) if QA_RESULTS else pl.DataFrame({"label": [], "status": [], "severity": [], "detail": []})
n_pass = verdict.filter(pl.col("status") == "PASS").height
n_warn = verdict.filter(pl.col("status") == "WARN").height
n_fail = verdict.filter(pl.col("status") == "FAIL").height

if n_fail > 0:
    overall = "FAIL"
elif n_warn > 0:
    overall = "WARN"
else:
    overall = "PASS"

print("=" * 70)
print(f"  QA VERDICT — overall: {overall}")
print(f"  PASS: {n_pass}   WARN: {n_warn}   FAIL: {n_fail}")
print("=" * 70)
if n_fail > 0:
    print("\nFailures:")
    for row in verdict.filter(pl.col("status") == "FAIL").to_dicts():
        print(f"  🔴 {row['label']}  —  {row['detail']}")
if n_warn > 0:
    print("\nWarnings:")
    for row in verdict.filter(pl.col("status") == "WARN").to_dicts():
        print(f"  🟠 {row['label']}  —  {row['detail']}")

verdict


  QA VERDICT — overall: WARN
  PASS: 4   WARN: 2   FAIL: 0

Warnings:
  🟠 |LVR_per_swap| tail count beyond p99.9 + 5σ  —  count=95 threshold=13088.97
  🟠 cumulative LVR within 5× of σ²/8 theoretical expectation  —  engine=15,567,599  theory=685,941  ratio=22.7×  σ̄=0.666  TVL̄=49,647,920


label,status,severity,detail
str,str,str,str
"""py_lvr_daily.lvr_total non-nul…","""PASS""","""warn""","""nulls=0"""
"""py_lvr_per_swap.lvr finite""","""PASS""","""fail""","""infinities=0"""
"""|LVR_per_swap| tail count beyo…","""WARN""","""warn""","""count=95 threshold=13088.97"""
"""py_lvr_per_swap.count() ≈ stg_…","""PASS""","""warn""","""stg=587406 lvr=584535 gap=2871…"
"""cumulative LVR within 5× of σ²…","""WARN""","""warn""","""engine=15,567,599 theory=685,…"
"""daily |LVR| scales with σ² (lo…","""PASS""","""warn""","""fitted_slope=0.80 (n=91)"""
